In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from pathlib import Path

In [ ]:
nlp = spacy.load("fr_core_news_lg", disable=["ner", "parser"])
nlp.max_length = 2_000_000

In [ ]:
dossier_path = Path("corpus_zola") # Chemin vers le dossier contenant les fichiers .txt

donnees = [] # Liste pour stocker les données extraites de chaque fichier

# On boucle sur tous les fichiers .txt du dossier
for fichier in dossier_path.glob("*.txt"):
    with open(fichier, "r", encoding="utf-8") as f:
        contenu = f.read()
        
        # On ajoute les données extraites du fichier à la liste
    donnees.append({
            "nom_fichier": fichier.name, # Nom du fichier
            "texte_brut": contenu, # Contenu brut du fichier
        })
      
         

# Création du tableau de bord (DataFrame)
df = pd.DataFrame(donnees)
print(f"{len(df)} textes chargés avec succès.")

In [ ]:
nb_tokens = df["texte_brut"].apply(lambda x: len(x.split())) # Calcul du nombre de tokens pour chaque texte
df["nb_tokens"] = nb_tokens # Ajout de la colonne "nb_tokens" au DataFrame
df

In [ ]:
df["nb_tokens"].describe()

In [ ]:
stop_word = {
    
    "je", "tu", "il", "elle", "ils", "elles", "nous", "vous",
    "me", "te", "se", "moi", "toi", "eux", "leur", "leurs", "ses",
    "mon", "ton", "son", "notre", "votre", "cette", "ça",
    "si", "même", "où", "dont", "sans", "quand", "bien", "là", "sous",
    "être", "avoir", "faire", "dire", "aller", "voir", "pouvoir",
    "savoir", "vouloir", "venir",
    "est", "ai", "as", "avez", "ont", "avaient", "étaient",
    "deux", "cent", "cents", "cinq", "dix", "vingt", "mille",
    "mme", "madame", " monsieur", "m", "mr",
    "nana", "gervaise", "louis", "lazare", "etienne", "claude", "suzanne", "mouret", "rougon", "macquart", "saccard", "lacroix",
}

In [ ]:
def nettoyer_texte(texte): #  Fonction pour nettoyer le texte : suppression des stop words, ponctuation, chiffres, espaces, et lemmatisation
    doc = nlp(texte) # Traiter le texte avec spaCy pour obtenir les tokens et leurs propriétés
    tokens = [] # Liste pour stocker les tokens nettoyés
    
    for token in doc:
        lemme = token.lemma_.lower() # Obtenir le lemme du token en minuscules pour une meilleure normalisation
        
        if (
            not token.is_stop # Ignorer les stop words (mots vides par défaut dans spaCy)
            and not token.is_punct # Ignorer la ponctuation
            and not token.like_num # Ignorer les chiffres
            and not token.is_space # Ignorer les espaces
            and token.pos_ in {"NOUN", "ADJ", "VERB"} 
            and len(token.lemma_) > 2 # Ne garder que les tokens dont la longueur du lemme est supérieure à 2 caractères
            and lemme not in stop_word # Ignorer les mots présents dans la liste personnalisée de stop words
        ):
            tokens.append(lemme) # Ajouter le lemme du token en minuscules à la liste des tokens nettoyés
    
    return " ".join(tokens) # Retourner le texte nettoyé en joignant les tokens avec des espaces

df["texte_nettoye"] = df["texte_brut"].apply(nettoyer_texte) # Application de la fonction de nettoyage à chaque segment de texte

df[["texte_brut", "texte_nettoye"]].head()

In [ ]:
nb_tokens = df["texte_nettoye"].apply(lambda x: len(x.split())) # Calcul du nombre de tokens pour chaque texte
df["nb_tokens"] = nb_tokens # Ajout de la colonne "nb_tokens" au DataFrame
df

In [ ]:
df.to_csv("corpus_zola_nettoye.csv", index=False)